# Прогноз золото-урановых зон: ML в основе, Random Forest без логистической регрессии

Эта версия использует **Random Forest как основную ML-модель**. Обучение строится на реальных точках и уверенных pseudo-labels из `geo_score`: верхние зоны используются как дополнительные положительные примеры, нижние — как фоновые отрицательные. Логистическая регрессия убрана, чтобы результат формировался RF, а сглаживание выполнялось отдельным пространственным фильтром.

## Запуск
Положи `Прогноз.zip` рядом с ноутбуком или запусти в папке проекта, где есть каталог `shp_dbf`. Основные параметры находятся в блоке `SETTINGS`.

In [1]:
import json
import os
import re
import shutil
import warnings
from pathlib import Path
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# =========================
# SETTINGS
# =========================
CELL_SIZE = 500
RANDOM_STATE = 42

# ML здесь является основой прогноза.
# Реальные точки дают подтвержденные положительные ячейки,
# а geo_score используется только для расширения обучающей выборки крайними псевдометками.
USE_SUPERVISED = True
USE_PSEUDO_LABELS = True
PSEUDO_POS_Q = 0.94   # верхние 6% geo_score => дополнительные положительные примеры, чуть расширяет синие зоны
PSEUDO_NEG_Q = 0.50   # нижние 50% geo_score => фоновые отрицательные примеры
MAX_NEG_PER_POS = 5   # балансировка: отрицательных не больше 5 на 1 положительный
MIN_POS_CELLS = 20

RF_N_ESTIMATORS = 800
RF_MAX_DEPTH = 8
RF_MIN_SAMPLES_LEAF = 10
RF_MIN_SAMPLES_SPLIT = 16

# В этой версии ML-основа — только Random Forest.
# Логистическая регрессия убрана: сложные сочетания факторов ловит RF,
# гладкость даем пространственным сглаживанием и мягкой геологической стабилизацией.
SMOOTH_PASSES = 4

# Итоговая карта строится по ML.
# Оставь 0.00, если нужно, чтобы ML был полностью в основе.
# Например, 0.10 даст мягкую геологическую стабилизацию: 90% ML + 10% geo_score.
GEO_STABILIZER_WEIGHT = 0.20

Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

N_DISPLAY_CLASSES = 20
SHOW_POINTS = False

# =========================
# PATHS
# =========================
def find_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base

    # Для запуска в ChatGPT/Colab: если рядом есть Прогноз.zip, распакуем автоматически.
    for zip_path in [Path("/mnt/data/Прогноз.zip"), Path.cwd() / "Прогноз.zip"]:
        if zip_path.exists():
            unzip_dir = Path("/mnt/data/prog_zip") if str(zip_path).startswith("/mnt/data") else Path.cwd() / "prog_zip"
            unzip_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(unzip_dir)
            for base in [unzip_dir, *unzip_dir.rglob("*")]:
                shp_dir = base / "shp_dbf"
                if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
                    return base

    raise FileNotFoundError("Не найден каталог с shp_dbf. Проверь BASE_DIR или распакуй Прогноз.zip.")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "ml_rf_pseudolabel_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "forecast_ml_rf_pseudolabel.gpkg"
OUT_PNG = OUT_DIR / "forecast_ml_rf_pseudolabel.png"
OUT_COMPARE = OUT_DIR / "compare_ml_rf_pseudolabel.png"
OUT_POINTS = OUT_DIR / "points_by_source.png"
OUT_PROX = OUT_DIR / "prox_magm_ml_rf_pseudolabel.png"
OUT_CSV = OUT_DIR / "grid_ml_rf_pseudolabel.csv"
OUT_JSON = OUT_DIR / "metrics_ml_rf_pseudolabel.json"

# =========================
# HELPERS
# =========================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    return (np.isfinite(arr) & (filled >= locmax))[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, _ = ndimage.label(arr, structure=structure)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0]) & (labeled > 0)
    return keep[rows, cols]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = d
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        t = np.sqrt(d)
    elif transform == "cbrt":
        t = np.cbrt(d)
    else:
        t = d
    scale = float(np.nanquantile(t, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(t)), 1.0)
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_points(mask_crs, aliases):
    base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}
    layers = []
    for name, shp_path in aliases.items():
        if name in base_names:
            continue
        gdf = to_crs_safe(load_layer(shp_path), mask_crs)
        types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in types or "MultiPoint" in types:
            gdf = gdf.copy()
            gdf["source_layer"] = name
            layers.append(gdf)
    if not layers:
        return None
    pts = pd.concat(layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def make_display_classes(grid):
    disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def mark_gold_zones(grid, shape, mask_union):
    q_best = float(grid["prospectivity"].quantile(0.97))
    q_support = float(grid["prospectivity"].quantile(0.88))
    q_local = float(grid["local_bonus"].quantile(0.97))
    q_coinc = float(grid["coincidence_score"].quantile(0.94))
    q_magm = float(grid["prox_magm"].quantile(0.90))
    q_tmagm = float(grid["tect_magm_intersection"].quantile(0.84))

    local_peak = local_max_mask(grid, "prospectivity", shape)
    mask_boundary = mask_union.boundary
    grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])

    support_zone = grid["prospectivity"] >= q_support

    core_gold = (
        support_zone &
        local_peak &
        (grid["prospectivity"] >= q_best) &
        (
            ((grid["local_bonus"] >= q_local) & (grid["tect_magm_intersection"] >= q_tmagm)) |
            ((grid["coincidence_score"] >= q_coinc) & (grid["prox_magm"] >= q_magm))
        )
    )

    linear_gold = (
        support_zone &
        (grid["prospectivity"] >= float(grid["prospectivity"].quantile(0.94))) &
        (grid["prox_magm"] >= q_magm) &
        (grid["tect_magm_intersection"] >= q_tmagm) &
        (grid["coincidence_score"] >= q_coinc)
    )

    edge_linear_gold = linear_gold & (grid["dist_to_boundary"] <= CELL_SIZE * 0.75)

    grid["gold_seed"] = (core_gold | edge_linear_gold).astype(int)
    grid["gold_zone"] = keep_large_components(grid, "gold_seed", shape, min_cells=4).astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    ax.legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    axes[1].legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_points_by_source(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.boundary.plot(ax=ax, linewidth=0.25, color="lightgray")
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        if "source_layer" in points.columns:
            points.plot(ax=ax, column="source_layer", categorical=True, legend=True, markersize=12)
        else:
            points.plot(ax=ax, color="red", markersize=12)
    set_mask_extent(ax, mask)
    ax.set_title("Точки, использованные как evidence для ML")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


# =========================
# LOAD DATA
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

# =========================
# GRID + FEATURES
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1)
    * np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1)
    * np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

# =========================
# GEO SCORE
# =========================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

# =========================
# ML MODEL: RANDOM FOREST + CONFIDENT PSEUDO-LABELS
# =========================
feature_cols = [
    "facies_prox", "paleo_prox", "struct_prox", "magm_prox", "tect1_prox", "tect2_prox",
    "facies_p", "paleo_p", "struct_p", "magm_p", "tect1_p", "tect2_p",
    "tect_best", "tect_mean", "factor_mean", "factor_min",
    "coincidence", "coincidence_soft", "local_bonus",
    "facies_x_paleo", "tect_x_paleo", "struct_x_magm", "tect_x_struct",
]
feature_cols = [c for c in feature_cols if c in grid.columns]

for c in feature_cols:
    grid[c] = grid[c].replace([np.inf, -np.inf], np.nan).fillna(0)

points = collect_points(mask.crs, aliases)
positive_cells_real = []
if points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")
    positive_cells_real = joined["cell_id"].dropna().astype(int).unique().tolist()

# target_ml:
#   1 = реальные точки + уверенно перспективные ячейки по geo_score
#   0 = уверенно низкие ячейки по geo_score
#   NaN = средние/неуверенные ячейки, не используются для обучения
# Так RF не пытается учиться на всех ячейках и меньше раздувает перспективные зоны.
grid["target_ml"] = np.nan
if len(positive_cells_real) > 0:
    grid.loc[grid["cell_id"].isin(positive_cells_real), "target_ml"] = 1

if USE_PSEUDO_LABELS:
    inside = grid["sed_mask"].astype(bool)
    q_pos = grid.loc[inside, "geo_score"].quantile(PSEUDO_POS_Q)
    q_neg = grid.loc[inside, "geo_score"].quantile(PSEUDO_NEG_Q)

    # берем только уверенные псевдометки
    grid.loc[inside & (grid["geo_score"] >= q_pos), "target_ml"] = 1
    grid.loc[inside & (grid["geo_score"] <= q_neg) & grid["target_ml"].isna(), "target_ml"] = 0

train = grid.dropna(subset=["target_ml"]).copy()
train["target_ml"] = train["target_ml"].astype(int)

# Балансировка: отрицательных не больше MAX_NEG_PER_POS на один положительный.
pos_train = train[train["target_ml"] == 1]
neg_train = train[train["target_ml"] == 0]
if len(pos_train) > 0 and len(neg_train) > len(pos_train) * MAX_NEG_PER_POS:
    neg_train = neg_train.sample(n=len(pos_train) * MAX_NEG_PER_POS, random_state=RANDOM_STATE)
    train = pd.concat([pos_train, neg_train], ignore_index=True)

use_ml = False
rf_test_proxy = None
ensemble_test_proxy = None
rf_auc = None
rf_ap = None
feature_importance = {}

X_train_full = train[feature_cols].fillna(0).to_numpy() if len(train) else np.empty((0, len(feature_cols)))
y_train_full = train["target_ml"].to_numpy() if len(train) else np.array([])

def _safe_metrics(y_true, prob):
    out = {"auc": None, "ap": None, "proxy": None}
    if len(np.unique(y_true)) > 1:
        try:
            out["auc"] = float(roc_auc_score(y_true, prob))
        except Exception:
            pass
        try:
            out["ap"] = float(average_precision_score(y_true, prob))
        except Exception:
            pass
        pos_mean = float(np.mean(prob[y_true == 1])) if np.any(y_true == 1) else np.nan
        neg_mean = float(np.mean(prob[y_true == 0])) if np.any(y_true == 0) else np.nan
        out["proxy"] = pos_mean - neg_mean
    return out

if USE_SUPERVISED and len(train) > 0:
    n_pos = int(np.sum(y_train_full == 1))
    n_neg = int(np.sum(y_train_full == 0))
    if n_pos >= MIN_POS_CELLS and n_neg > 0:
        # Оценка качества — только прокси, потому что pseudo-labels не являются истинной геологической проверкой.
        if min(n_pos, n_neg) >= 5:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_train_full, y_train_full, test_size=0.25,
                random_state=RANDOM_STATE, stratify=y_train_full
            )
            rf_eval = RandomForestClassifier(
                n_estimators=RF_N_ESTIMATORS,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_SAMPLES_LEAF,
                min_samples_split=RF_MIN_SAMPLES_SPLIT,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            rf_eval.fit(X_tr, y_tr)
            rf_eval_prob = rf_eval.predict_proba(X_te)[:, 1]
            m = _safe_metrics(y_te, rf_eval_prob)
            rf_auc, rf_ap, rf_test_proxy = m["auc"], m["ap"], m["proxy"]
            ensemble_test_proxy = rf_test_proxy

        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        rf.fit(X_train_full, y_train_full)
        X_all = grid[feature_cols].fillna(0).to_numpy()
        grid["rf_score"] = robust_normalize_01(rf.predict_proba(X_all)[:, 1], 0.02, 0.98)
        feature_importance = dict(zip(feature_cols, rf.feature_importances_.tolist()))
        use_ml = True
    else:
        grid["rf_score"] = grid["geo_score"]
else:
    grid["rf_score"] = grid["geo_score"]

if use_ml:
    grid["ml_score"] = grid["rf_score"]
else:
    grid["ml_score"] = grid["geo_score"]

grid["ml_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "ml_score", grid_shape, passes=SMOOTH_PASSES), 0.02, 0.98)

# =========================
# FINAL SURFACE
# =========================
# local_bonus остается только как вспомогательное поле для выделения компактных желтых зон,
# но НЕ добавляется вручную в итоговый score.
grid["local_bonus"] = robust_normalize_01(
    0.38 * grid["tect_intersection"] +
    0.37 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"],
    0.02, 0.98,
)

# Итог: ML в основе: основной слой — ансамбль RF+LogReg, geo_score только стабилизирует форму карты.
ml_part = grid["ml_score_sm"].to_numpy()
geo_part = grid["geo_score_sm"].to_numpy()
w_geo = float(np.clip(GEO_STABILIZER_WEIGHT, 0, 1))
grid["prospectivity_raw"] = (1.0 - w_geo) * ml_part + w_geo * geo_part
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=2)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)
grid["prognoz"] = 1.0 - grid["prospectivity"]

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

# =========================
# SAVE
# =========================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_points_by_source(grid, mask, points, OUT_POINTS)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "som_used": False,
    "kmeans_used": False,
    "models_used": ["RandomForest"],
    "smooth_passes": SMOOTH_PASSES,
    "ml_is_main_model": True,
    "geo_stabilizer_weight": GEO_STABILIZER_WEIGHT,
    "real_positive_cells": int(len(positive_cells)),
    "train_positive_cells": int((grid["target_ml"] == 1).sum()),
    "train_negative_cells": int((grid["target_ml"] == 0).sum()),
    "pseudo_pos_quantile": PSEUDO_POS_Q,
    "pseudo_neg_quantile": PSEUDO_NEG_Q,
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "rf_test_proxy": None if rf_test_proxy is None else float(rf_test_proxy),
    "ensemble_test_proxy": None if ensemble_test_proxy is None else float(ensemble_test_proxy),
    "rf_auc_proxy": rf_auc,
    "ensemble_auc_proxy": ensemble_auc,
    "rf_average_precision_proxy": rf_ap,
    "ensemble_average_precision_proxy": ensemble_ap,
    "point_count": int(len(points)) if points is not None else 0,
    "rf_feature_importance": feature_importance,
}
OUT_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"POINTS: {OUT_POINTS}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("Реальные positive cells:", len(positive_cells))
print("Train positives:", int((grid["target_ml"] == 1).sum()), "Train negatives:", int((grid["target_ml"] == 0).sum()))
print("RF proxy:", rf_test_proxy)
print(grid[["prospectivity", "prognoz", "rf_score_sm", "ml_score_sm", "geo_score_sm", "gold_zone"]].describe())


KeyError: 'sed_mask'